In [11]:
import torch
import numpy as np
from refinenet import RefineNet
from collections import OrderedDict

In [12]:
refiner_model = './weights/craft_refiner_CTW1500.pth'

In [13]:
def copyStateDict(state_dict):
    if list(state_dict.keys())[0].startswith("module"):
        start_idx = 1
    else:
        start_idx = 0
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        name = ".".join(k.split(".")[start_idx:])
        new_state_dict[name] = v
    return new_state_dict

In [14]:
refine_net = RefineNet()
refine_net.load_state_dict(copyStateDict(torch.load(refiner_model, map_location = 'cpu')))
refine_net.eval()

RefineNet(
  (last_conv): Sequential(
    (0): Conv2d(34, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU(inplace=True)
  )
  (aspp1): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(6, 6), dilation=(6, 6))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(128, 128, kernel_size=(1, 1), stride=(1, 1))
    (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=Tru

In [15]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

input1 = torch.randn(1,64,32,2).to(device)
input2 = torch.randn(1,32,64,32).to(device)

dynamic_axes = {"input1" : {1 : "width", 2 : "height"}, "input2" : {2 : "width", 3 : "height"}}

In [16]:
torch.onnx.export(refine_net,
                 (input1, input2),
                 "./weights/refine_net.onnx",
                 opset_version=11,
                 input_names = ['input1','input2'],
                 output_names = ['output'],
                 dynamic_axes = dynamic_axes)